In [0]:
dbutils.widgets.text("ConfigId", "11")
ConfigId = dbutils.widgets.get("ConfigId")

dbutils.widgets.text("pipeLineId", "1")
pipeLineId = dbutils.widgets.get("pipeLineId")

dbutils.widgets.text("SourceTypeId", "6")
SourceTypeId = dbutils.widgets.get("SourceTypeId")

dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')
DestinationContainerPath=ExternalLocation_raw
DestinationFolderPath=ExternalLocation_raw+DestinationContainerPath


dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePrefix = '/dbfs/'+ExternalLocationName

# dbutils.widgets.text("DestinationContainerPath", "raw")
# DestinationContainerPath = dbutils.widgets.get("DestinationContainerPath")

# dbutils.widgets.text("DestinationFolderPath", "DeviceTwin")
# DestinationFolderPath = dbutils.widgets.get("DestinationFolderPath")

dbutils.widgets.text("DestinationFileName", "DeviceTwin_<TimeStamp>.json")
DestinationFileName = dbutils.widgets.get("DestinationFileName")

dbutils.widgets.text("InvalidFilePathFormat", "")
InvalidFilePathFormat = dbutils.widgets.get("InvalidFilePathFormat")

In [0]:
%run ../Configurations/Init_Scripts

In [0]:
%sh pip install azure-iot-hub

In [0]:
spark.conf.set("spark.sql.catalog.current", CatalogName)

In [0]:
import json
from azure.iot.hub import IoTHubRegistryManager
from azure.iot.hub.protocol.models import QuerySpecification


In [0]:
# using datetime module
import datetime;
 
# ct stores current time
currentTime = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
DestinationFileName = DestinationFileName.replace('<TimeStamp>',currentTime)
print(DestinationFolderPath)

In [0]:
processedFileList = []
deviceTwinArray = []
twininfo = []
test = []
fileDetails = dict()

# filePathPrefix = '/dbfs/mnt'
destinationPath = filePathPrefix+'/'+DestinationContainerPath+'/'+DestinationFolderPath+'/'+DestinationFileName

fileDetails['SourceFolderPath'] = ''
fileDetails['SourceFileName'] = ''
fileDetails['SourceContainerPath'] = ''
fileDetails['DestinationFolderPath'] = DestinationFolderPath
fileDetails['DestinationFileName'] = DestinationFileName
fileDetails['DestinationContainerPath'] = DestinationContainerPath
fileDetails['LoadType'] = 'DeviceTwinFile'

fileDetails['PipelineRunId'] = pipeLineId
fileDetails['ConfigId'] = ConfigId
fileDetails['SourceTypeId'] = SourceTypeId

    
IOTHubConnectionString = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="IoTHubConnectionString")


In [0]:
try:
    print('Device twin data load started')
    registry_manager = IoTHubRegistryManager(IOTHubConnectionString)
    query = QuerySpecification(query= "SELECT * FROM devices")
    
    results = registry_manager.query_iot_hub(query)
    twininfo = results.items
    continuationToken = results.continuation_token
    while(continuationToken is not None):
        results = registry_manager.query_iot_hub(query,continuation_token=continuationToken)
        twininfo = twininfo+results.items
        continuationToken = results.continuation_token
    print('Device twin data load end')
    print('Device twin data read from registry and load into json array started')
    for twin in twininfo:
        devicetwin = dict()
        twindata = twin.as_dict()
        device_id = twindata.get('device_id')
        productType = twindata.get('properties').get('reported').get('productType')
        if(productType is None):
            if(device_id.startswith( 'CT' )):
                productType = 'CT'
            elif(device_id.startswith( 'RS' )):
                productType = 'RS'
            else:
                productType = 'COM3'

        devicetwin['twinData'] = twindata
        devicetwin['DeviceId'] = device_id
        devicetwin['ProductType'] = productType
        deviceTwinArray.append(devicetwin)
        test.append(twindata)
    print('Device twin data read from registry and load into json array started')
    print('Load data to json file started')

    with open(destinationPath, "w") as outfile:
        outfile.write(json.dumps(deviceTwinArray))
    print('Load data to json file end')

    fileDetails['PipelineStatus'] = 'Succeeded'
    processedFileList.append(fileDetails)
    logIntoIngestionLogTable(processedFileList,'ADB_LoadDeviceTwin')
except Exception as e:
    fileDetails['PipelineStatus'] = 'Failed'
    fileDetails['ErrorMessage'] = str(e)
    processedFileList.append(fileDetails)
    logIntoIngestionLogTable(processedFileList,'ADB_LoadDeviceTwin')    
    raise